# Elastic Stack Setup Guide
## Windows (VM) | macOS | Linux — CLI Reference & Comparison

> **Author:** Long Chung  
> **Stack:** Elasticsearch 7.16.3 + Kibana 7.16.3  
> **Environment:** macOS M4 Air → VMware Fusion → Windows 11 ARM64

---

## Table of Contents
1. [Environment Setup](#1-environment-setup)
2. [Windows VM Setup on Mac M4](#2-windows-vm-setup-on-mac-m4)
3. [Elasticsearch Installation](#3-elasticsearch-installation)
4. [Kibana Installation](#4-kibana-installation)
5. [CLI Comparison: Mac vs Windows vs Linux](#5-cli-comparison)
6. [Service Management](#6-service-management)
7. [Elasticsearch API Quick Reference](#7-elasticsearch-api-quick-reference)
8. [Troubleshooting](#9-troubleshooting)
9. [Daily Workflow](#10-daily-workflow)

---

## 1. Environment Setup

### Hardware
```
Host:    MacBook Air M4 (Apple Silicon ARM64)
RAM:     16GB
OS:      macOS Sequoia
VM Tool: VMware Fusion 13
```

### VM Configuration
```
Guest OS:   Windows 11 Pro ARM64 (Build 26200)
CPU:        2 cores
RAM:        4096 MB
Disk:       34.4 GB
Network:    NAT (Share with Mac)
IP:         192.168.19.131
```

### Software Stack
```
Elasticsearch:  7.16.3
Kibana:         7.16.3
VS Code:        1.112.0 (ARM64)
Git:            2.53.0
```

---

## 2. Windows VM Setup on Mac M4

### Prerequisites
- VMware Fusion 13+ (supports Apple Silicon)
- Windows 11 ARM64 ISO (`Win11_25H2_English_Arm64.iso`)
- Get ISO from: https://aka.ms/windowsinsiderpreview

### VMware Fusion Configuration
```
File → New → Install from disc or image
→ Select Win11_25H2_English_Arm64.iso
→ Firmware: UEFI + Secure Boot ✅
→ CPU: 2-4 cores
→ RAM: 4096-8192 MB
→ Disk: 60-80 GB (dynamic)
→ Network: Share with my Mac (NAT)
```

### Fix Boot Order (prevent CD boot loop)
Edit `.vmx` file on Mac:
```bash
open -a TextEdit "/Users/YOUR_USERNAME/Virtual Machines.localized/Windows 11 64-bit Arm.vmwarevm/Windows 11 64-bit Arm.vmx"
```
Change:
```
bios.bootOrder = "CDROM"   # ← change this
bios.bootOrder = "hdd"     # ← to this
```
Also add for nested virtualization attempt:
```
vhv.enable = "TRUE"
```

### VMware Tools Installation
```
Virtual Machine → Install VMware Tools
→ Inside Windows: Run setup.exe from CD Drive
→ Choose: Repair (if already installed)
→ Restart Windows
```
Confirms working when:
- ✅ Mouse moves freely in/out of VM
- ✅ Copy/paste works between Mac and VM
- ✅ Resolution auto-adjusts

### Essential Windows Apps
```powershell
# Install via winget (Windows package manager)
winget install Google.Chrome
winget install Microsoft.VisualStudioCode
winget install Git.Git
winget install Elastic.Elasticsearch
```

### Git Configuration
```powershell
git config --global user.name "Your Name"
git config --global user.email "your@email.com"
git config --global core.editor "code --wait"

# Verify
git config --list
```

---

## 3. Elasticsearch Installation

### Windows (winget)
```powershell
winget install Elastic.Elasticsearch

# Verify service is running
Get-Service elasticsearch*

# Test
curl http://localhost:9200
```

### macOS (Homebrew)
```bash
brew tap elastic/tap
brew install elastic/tap/elasticsearch-full

# Start
brew services start elasticsearch

# Test
curl http://localhost:9200
```

### Linux (apt - Ubuntu/Debian)
```bash
# Add GPG key
wget -qO - https://artifacts.elastic.co/GPG-KEY-elasticsearch | sudo apt-key add -

# Add repository
sudo apt-get install apt-transport-https
echo "deb https://artifacts.elastic.co/packages/7.x/apt stable main" | sudo tee /etc/apt/sources.list.d/elastic-7.x.list

# Install
sudo apt-get update && sudo apt-get install elasticsearch

# Start
sudo systemctl start elasticsearch
sudo systemctl enable elasticsearch

# Test
curl http://localhost:9200
```

---

## 4. Kibana Installation

### Windows (manual ZIP - recommended for 7.x)
```powershell
# Download from:
# https://www.elastic.co/downloads/past-releases/kibana-7-16-3
# → WINDOWS ZIP

# Extract to short path (avoid path too long error!)
# Open PowerShell as Administrator:
cd "C:\Users\YOUR_USERNAME\Downloads"
Expand-Archive -Path "kibana-7.16.3-windows-x86_64.zip" -DestinationPath "C:\kibana"

# Start Kibana
cd "C:\kibana\kibana-7.16.3-windows-x86_64\bin"
.\kibana.bat

# Access UI
# Open browser → http://localhost:5601
```

### macOS (Homebrew)
```bash
brew install elastic/tap/kibana-full
brew services start kibana

# Access UI
open http://localhost:5601
```

### Linux (apt)
```bash
sudo apt-get install kibana
sudo systemctl start kibana
sudo systemctl enable kibana

# Access UI
# Open browser → http://localhost:5601
```

---

## 5. CLI Comparison

### Package Management
| Task | macOS | Windows | Linux |
|------|-------|---------|-------|
| Install package | `brew install X` | `winget install X` | `sudo apt install X` |
| Remove package | `brew uninstall X` | `winget uninstall X` | `sudo apt remove X` |
| Update all | `brew upgrade` | `winget upgrade --all` | `sudo apt upgrade` |
| Search package | `brew search X` | `winget search X` | `apt search X` |

### File System
| Task | macOS/Linux | Windows PowerShell |
|------|-------------|-------------------|
| List files | `ls -la` | `Get-ChildItem -Force` |
| Change dir | `cd /path/to/dir` | `cd "C:\path\to\dir"` |
| Home dir | `cd ~` | `cd $HOME` |
| Print dir | `pwd` | `Get-Location` |
| Make dir | `mkdir folder` | `mkdir folder` |
| Remove dir | `rm -rf folder` | `Remove-Item -Recurse folder` |
| Copy file | `cp file dest` | `Copy-Item file dest` |
| Move file | `mv file dest` | `Move-Item file dest` |
| View file | `cat file.txt` | `Get-Content file.txt` |
| Edit file | `nano file.txt` | `notepad file.txt` or `code file.txt` |
| Find files | `find / -name "*.yml"` | `Get-ChildItem -Recurse -Filter *.yml` |
| Search text | `grep -r "text" .` | `Select-String -Recurse "text"` |

### Process & Network
| Task | macOS/Linux | Windows PowerShell |
|------|-------------|-------------------|
| Check port | `lsof -i :9200` | `netstat -ano \| findstr :9200` |
| Kill process | `kill -9 PID` | `taskkill /PID 1234 /F` |
| List processes | `ps aux` | `Get-Process` |
| Check IP | `ifconfig` | `ipconfig` |
| Ping | `ping 8.8.8.8` | `ping 8.8.8.8` |
| Curl | `curl http://localhost:9200` | `curl http://localhost:9200` |

### Environment Variables
| Task | macOS/Linux | Windows PowerShell |
|------|-------------|-------------------|
| Set variable | `export VAR=value` | `$env:VAR = "value"` |
| View variable | `echo $VAR` | `echo $env:VAR` |
| Permanent set | Add to `~/.zshrc` | System → Environment Variables |
| Apply changes | `source ~/.zshrc` | Restart PowerShell |

### Tail Logs
| Task | macOS/Linux | Windows PowerShell |
|------|-------------|-------------------|
| Follow log | `tail -f logfile.log` | `Get-Content logfile.log -Wait` |
| Last 100 lines | `tail -n 100 logfile.log` | `Get-Content logfile.log -Tail 100` |

---

## 6. Service Management

### Windows
```powershell
# Elasticsearch
net start elasticsearch        # start
net stop elasticsearch         # stop
Get-Service elasticsearch*     # status

# Kibana (run manually)
cd "C:\kibana\kibana-7.16.3-windows-x86_64\bin"
.\kibana.bat                   # start (keep window open)
# Ctrl+C to stop
```

### macOS
```bash
brew services start elasticsearch
brew services stop elasticsearch
brew services restart elasticsearch
brew services start kibana
brew services stop kibana
brew services list             # check all services
```

### Linux
```bash
sudo systemctl start elasticsearch
sudo systemctl stop elasticsearch
sudo systemctl restart elasticsearch
sudo systemctl status elasticsearch
sudo systemctl enable elasticsearch   # auto-start on boot

sudo systemctl start kibana
sudo systemctl stop kibana
sudo systemctl status kibana
```

---

## 7. Elasticsearch API Quick Reference

These commands work identically on all platforms:

```bash
# Cluster health
curl http://localhost:9200/_cluster/health?pretty

# List all indices
curl http://localhost:9200/_cat/indices?v

# List nodes
curl http://localhost:9200/_cat/nodes?v

# Create index
curl -X PUT http://localhost:9200/my-index

# Index a document
curl -X POST http://localhost:9200/my-index/_doc \
  -H "Content-Type: application/json" \
  -d '{"name": "test", "message": "hello elasticsearch"}'

# Search all documents
curl http://localhost:9200/my-index/_search?pretty

# Search with query
curl -X GET http://localhost:9200/my-index/_search \
  -H "Content-Type: application/json" \
  -d '{"query": {"match": {"message": "hello"}}}'

# Delete index
curl -X DELETE http://localhost:9200/my-index

# Cluster settings
curl http://localhost:9200/_cluster/settings?pretty

# Index mapping
curl http://localhost:9200/my-index/_mapping?pretty
```

---

## 8. Troubleshooting

### Elasticsearch Won't Start (Windows)
```powershell
# Check logs
Get-Content "C:\ProgramData\Elastic\Elasticsearch\logs\elasticsearch.log" -Tail 50

# Check Java
java -version

# Check port conflict
netstat -ano | findstr :9200

# Restart service
net stop elasticsearch
net start elasticsearch
```

### Elasticsearch Won't Start (Linux/Mac)
```bash
# Check logs
tail -f /var/log/elasticsearch/elasticsearch.log  # Linux
tail -f /usr/local/var/log/elasticsearch/elasticsearch.log  # Mac

# Check port
lsof -i :9200

# Memory issue - edit JVM options
sudo nano /etc/elasticsearch/jvm.options
# Set: -Xms1g -Xmx1g
```

### Kibana "Degraded" Status
```
This is NORMAL — Kibana does frequent health checks
"degraded → available" cycling = working correctly
Only worry if it stays degraded for 5+ minutes
```

### Windows Path Too Long Error (0x80010135)
```powershell
# Always extract to short paths!
# WRONG: C:\Users\Long Chung\Downloads\kibana-7.16.3\...
# RIGHT: C:\kibana\

Expand-Archive -Path "kibana.zip" -DestinationPath "C:\kibana"
```

### Network Not Connected in VM
```powershell
# Reset network stack
netsh winsock reset
netsh int ip reset
ipconfig /release
ipconfig /renew

# Verify
ping 8.8.8.8
```

---

## 9. Daily Workflow

### Start Elastic Stack (Windows)
```powershell
# 1. Start Elasticsearch (runs as service, auto-starts)
net start elasticsearch

# 2. Start Kibana
cd "C:\kibana\kibana-7.16.3-windows-x86_64\bin"
.\kibana.bat

# 3. Wait 60 seconds, then open browser
# http://localhost:5601

# 4. Verify ES
curl http://localhost:9200/_cluster/health?pretty
```

### Start Elastic Stack (macOS)
```bash
brew services start elasticsearch
brew services start kibana
open http://localhost:5601
```

### Start Elastic Stack (Linux)
```bash
sudo systemctl start elasticsearch
sudo systemctl start kibana
# Open browser → http://localhost:5601
```

### Keyboard Shortcuts: Mac → Windows
| Action | Mac | Windows |
|--------|-----|---------|
| Copy | `Cmd+C` | `Ctrl+C` |
| Paste | `Cmd+V` | `Ctrl+V` |
| Cut | `Cmd+X` | `Ctrl+X` |
| Undo | `Cmd+Z` | `Ctrl+Z` |
| Find | `Cmd+F` | `Ctrl+F` |
| Save | `Cmd+S` | `Ctrl+S` |
| New tab | `Cmd+T` | `Ctrl+T` |
| Switch apps | `Cmd+Tab` | `Alt+Tab` |
| Spotlight/Search | `Cmd+Space` | `Win key` |
| Force quit | `Cmd+Option+Esc` | `Ctrl+Shift+Esc` |
| Lock screen | `Cmd+Ctrl+Q` | `Win+L` |
| Screenshot | `Cmd+Shift+4` | `Win+Shift+S` |
| VS Code palette | `Cmd+Shift+P` | `Ctrl+Shift+P` |

---

## File Locations Quick Reference

### Windows
```
ES Config:      C:\ProgramData\Elastic\Elasticsearch\config\elasticsearch.yml
ES Logs:        C:\ProgramData\Elastic\Elasticsearch\logs\
ES Data:        C:\ProgramData\Elastic\Elasticsearch\data\
Kibana Config:  C:\kibana\kibana-7.16.3-windows-x86_64\config\kibana.yml
Kibana Logs:    C:\kibana\kibana-7.16.3-windows-x86_64\logs\
Kibana Bin:     C:\kibana\kibana-7.16.3-windows-x86_64\bin\kibana.bat
```

### macOS
```
ES Config:      /usr/local/etc/elasticsearch/elasticsearch.yml
ES Logs:        /usr/local/var/log/elasticsearch/
ES Data:        /usr/local/var/lib/elasticsearch/
Kibana Config:  /usr/local/etc/kibana/kibana.yml
Kibana Logs:    /usr/local/var/log/kibana/
```

### Linux
```
ES Config:      /etc/elasticsearch/elasticsearch.yml
ES Logs:        /var/log/elasticsearch/
ES Data:        /var/lib/elasticsearch/
Kibana Config:  /etc/kibana/kibana.yml
Kibana Logs:    /var/log/kibana/
```

---

